# ClinVar ALS Pathogenic Check

This notebook does two things:
1. Download ClinVar VCF files (GRCh38)
2. Check variants in `mutation_summary_table.csv` for ClinVar pathogenic annotation related to ALS


In [ ]:
# system("wget ftp://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/clinvar.vcf.gz")
# system("wget ftp://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/clinvar.vcf.gz.tbi")


In [ ]:
clinvar_gz_path <- "clinvar.vcf.gz"
clinvar_vcf_path <- "clinvar.vcf"

if (!file.exists(clinvar_gz_path)) {
  stop(sprintf("Cannot find file: %s", clinvar_gz_path))
}

if (!file.exists(clinvar_vcf_path)) {
  cmd <- sprintf("gunzip -c %s > %s", shQuote(clinvar_gz_path), shQuote(clinvar_vcf_path))
  status <- system(cmd)
  if (status != 0 || !file.exists(clinvar_vcf_path)) {
    stop("Failed to decompress clinvar.vcf.gz")
  }
  cat(sprintf("Decompressed to: %s\n", clinvar_vcf_path))
} else {
  cat(sprintf("Already exists: %s\n", clinvar_vcf_path))
}


In [ ]:
# ---- 4) Build participant-level carrier tables ----
# Two versions are produced:
#   Version 1 (ALS-pathogenic): mutations that are pathogenic AND ALS-related (is_pathogenic_als == TRUE)
#   Version 2 (any pathogenic/likely pathogenic): mutations that are pathogenic or likely pathogenic
#              regardless of disease association (is_pathogenic == TRUE)

als_mutation_file <- "mutation_summary_table_with_clinvar_als_pathogenic.csv"
raw_dir <- "gene_extraction_results"

# --- Version 1: ALS-pathogenic ---
out_wide_als  <- "eid_carrier_status_als_pathogenic_mutations.csv"
out_any_als   <- "eid_carrier_any_als_pathogenic_mutations.csv"

# --- Version 2: any pathogenic/likely pathogenic ---
out_wide_any_path  <- "eid_carrier_status_any_pathogenic_mutations.csv"
out_any_any_path   <- "eid_carrier_any_pathogenic_mutations.csv"

if (!file.exists(als_mutation_file)) {
  stop(sprintf("Cannot find %s. Run the previous analysis cell first.", als_mutation_file))
}
if (!dir.exists(raw_dir)) {
  stop(sprintf("Cannot find directory: %s", raw_dir))
}

als_df <- read.csv(als_mutation_file, stringsAsFactors = FALSE, check.names = FALSE)
need_cols <- c("gene", "mutation", "is_pathogenic_als", "is_pathogenic")
missing_cols <- setdiff(need_cols, names(als_df))
if (length(missing_cols) > 0) {
  stop(sprintf("Missing required columns in %s: %s", als_mutation_file, paste(missing_cols, collapse = ", ")))
}

# ---- helper: build wide + any carrier tables for a given mutation set ----
build_carrier_tables <- function(targets, raw_dir) {
  targets     <- unique(targets)
  target_muts <- unique(targets$mutation)
  carrier_map <- list()
  all_eids    <- character(0)

  for (gene_name in unique(targets$gene)) {
    raw_file <- file.path(raw_dir, sprintf("UKB_%s_pathogenic_corrected.raw", gene_name))
    if (!file.exists(raw_file)) {
      warning(sprintf("Raw file not found for gene '%s': %s", gene_name, raw_file))
      next
    }

    df <- utils::read.table(
      raw_file, sep = "\t", header = TRUE, quote = "",
      comment.char = "", fill = TRUE, check.names = FALSE,
      stringsAsFactors = FALSE
    )

    if (!"IID" %in% names(df)) {
      warning(sprintf("IID column not found in %s; skip", raw_file))
      next
    }

    eids     <- as.character(df$IID)
    all_eids <- union(all_eids, eids)

    gene_mutations <- unique(targets$mutation[targets$gene == gene_name])

    for (mut in gene_mutations) {
      if (!mut %in% names(df)) {
        warning(sprintf("Mutation column missing in %s: %s", basename(raw_file), mut))
        next
      }
      x       <- suppressWarnings(as.numeric(df[[mut]]))
      carrier <- ifelse(!is.na(x) & x > 0, 1L, 0L)
      v        <- carrier
      names(v) <- eids
      carrier_map[[mut]] <- v
    }
  }

  if (length(all_eids) == 0) stop("No participant IDs (IID) found from raw files.")

  # Wide table: eid + one column per mutation (0/1)
  wide_df <- data.frame(eid = all_eids, stringsAsFactors = FALSE)
  for (mut in target_muts) {
    col <- integer(length(all_eids))
    if (!is.null(carrier_map[[mut]])) {
      idx <- match(all_eids, names(carrier_map[[mut]]))
      hit <- !is.na(idx)
      col[hit] <- carrier_map[[mut]][idx[hit]]
    }
    wide_df[[mut]] <- as.integer(col)
  }

  # Any-carrier table: eid + single binary column
  wide_mat  <- as.matrix(wide_df[, -1, drop = FALSE])
  any_flag  <- as.integer(rowSums(wide_mat, na.rm = TRUE) > 0)
  list(wide = wide_df, any_flag = any_flag)
}

# ============================================================
# Version 1: Participants carrying >=1 ALS-pathogenic mutation
# (pathogenic/likely-pathogenic AND ALS-related disease in ClinVar)
# ============================================================
cat("\n=== Version 1: ALS-pathogenic mutations ===\n")
targets_als <- als_df[als_df$is_pathogenic_als, c("gene", "mutation")]
if (nrow(targets_als) == 0) stop("No ALS-pathogenic mutations found (is_pathogenic_als == TRUE).")

res_als <- build_carrier_tables(targets_als, raw_dir)

n_als_muts <- ncol(res_als$wide) - 1
any_df_als <- data.frame(
  eid             = res_als$wide$eid,
  carrier_any_als = res_als$any_flag,
  stringsAsFactors = FALSE
)

write.csv(res_als$wide, out_wide_als, row.names = FALSE)
write.csv(any_df_als,   out_any_als,  row.names = FALSE)

cat(sprintf("ALS-pathogenic mutations used        : %d\n", n_als_muts))
cat(sprintf("Participants in wide table           : %d\n", nrow(res_als$wide)))
cat(sprintf("Participants carrying >=1 ALS-pathogenic mutation: %d\n", sum(any_df_als$carrier_any_als)))
cat(sprintf("Saved wide : %s\n", out_wide_als))
cat(sprintf("Saved any  : %s\n", out_any_als))

# ============================================================
# Version 2: Participants carrying >=1 pathogenic or likely
# pathogenic mutation (any disease, not restricted to ALS)
# ============================================================
cat("\n=== Version 2: Any pathogenic/likely-pathogenic mutations ===\n")
targets_path <- als_df[als_df$is_pathogenic, c("gene", "mutation")]
if (nrow(targets_path) == 0) stop("No pathogenic/likely-pathogenic mutations found (is_pathogenic == TRUE).")

res_path <- build_carrier_tables(targets_path, raw_dir)

n_path_muts <- ncol(res_path$wide) - 1
any_df_path <- data.frame(
  eid              = res_path$wide$eid,
  carrier_any_path = res_path$any_flag,
  stringsAsFactors = FALSE
)

write.csv(res_path$wide, out_wide_any_path, row.names = FALSE)
write.csv(any_df_path,   out_any_any_path,  row.names = FALSE)

cat(sprintf("Any-pathogenic mutations used         : %d\n", n_path_muts))
cat(sprintf("Participants in wide table            : %d\n", nrow(res_path$wide)))
cat(sprintf("Participants carrying >=1 pathogenic/likely-pathogenic mutation: %d\n", sum(any_df_path$carrier_any_path)))
cat(sprintf("Saved wide : %s\n", out_wide_any_path))
cat(sprintf("Saved any  : %s\n", out_any_any_path))
